# Baseline classifier — load and train

Loads `starter/baseline/baseline_classifier.py` and trains it on `starter/data/train.csv`.

In [1]:
import importlib.util
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODULE_PATH = Path(f"{PROJECT_ROOT}/starter/baseline/baseline_classifier.py")
DATA_PATH = Path(f"{PROJECT_ROOT}/starter/data/train.csv")

assert MODULE_PATH.exists(), MODULE_PATH
assert DATA_PATH.exists(), DATA_PATH

In [2]:
# Load the script as a module without needing it on sys.path.
spec = importlib.util.spec_from_file_location("baseline_classifier", MODULE_PATH)
baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print(baseline.ROUTES)

['account-access', 'transaction-dispute', 'fraud-report', 'general']


## Data

In [3]:
from collections import Counter

texts, labels = baseline.load(DATA_PATH)
print(f"{len(texts)} rows")
Counter(labels)

400 rows


Counter({'general': 160,
         'account-access': 100,
         'transaction-dispute': 90,
         'fraud-report': 50})

## Train

Runs the module's own `main()`, which vectorises, splits 80/20 and reports accuracy.

In [4]:
acc = baseline.main()
acc

loaded 400 rows
test accuracy: 0.9875


0.9875

## Same pipeline, reproduced for inspection

`main()` returns only accuracy, so rebuild it here to reach the fitted objects.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X = vectorizer.fit_transform(texts)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=0
)

clf = LogisticRegression(max_iter=2000, C=10.0)
clf.fit(X_train, y_train)

print(f"{X.shape[1]} features, {X_train.shape[0]} train / {X_test.shape[0]} test")

1522 features, 320 train / 80 test


In [6]:
preds = clf.predict(X_test)

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, preds)

print(f"train accuracy: {train_acc:.4f}")
print(f"test accuracy:  {test_acc:.4f}")
print(classification_report(y_test, preds, zero_division=0))

train accuracy: 1.0000
test accuracy:  0.9875
                     precision    recall  f1-score   support

     account-access       1.00      1.00      1.00        16
       fraud-report       1.00      0.93      0.97        15
            general       0.97      1.00      0.99        35
transaction-dispute       1.00      1.00      1.00        14

           accuracy                           0.99        80
          macro avg       0.99      0.98      0.99        80
       weighted avg       0.99      0.99      0.99        80



In [7]:
labels_sorted = sorted(set(labels))
cm = confusion_matrix(y_test, preds, labels=labels_sorted)

print("rows = true, cols = predicted")
print(labels_sorted)
print(cm)

rows = true, cols = predicted
['account-access', 'fraud-report', 'general', 'transaction-dispute']
[[16  0  0  0]
 [ 0 14  1  0]
 [ 0  0 35  0]
 [ 0  0  0 14]]
